[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itorque2024/capstone-ninjavan/blob/main/notebooks/02_predictive_maintenance.ipynb)

# 02 — Predictive Maintenance
**NinjaVan Operations Intelligence Suite**

XGBoost classifier — predicts vehicle failure risk 48–72 h in advance.

---
- **Run in:** Google Colab (CPU is fine)
- **Input:** `ninjavan_optionB_datasets/maintenance_data.csv`
- **Output:** `src/models/maintenance_model.pkl` → used by `src/agents/maintenance_agent.py`
- **AI types covered:** ML (XGBoost classifier), Optimization (SMOTE class balancing)


In [ ]:
# Install packages not available by default on Colab
!pip install -q xgboost imbalanced-learn scikit-learn pandas numpy matplotlib seaborn joblib


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


In [ ]:
import os

if not os.path.exists('capstone-ninjavan'):
    !git clone https://github.com/itorque2024/capstone-ninjavan.git

if os.path.basename(os.getcwd()) != 'capstone-ninjavan':
    os.chdir('capstone-ninjavan')

print('Working dir:', os.getcwd())

## 1. Load & Explore Data

In [ ]:
raw = pd.read_csv('ninjavan_optionB_datasets/maintenance_data.csv')
print(f'Shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')
print(f'Missing: {raw.isnull().sum().to_dict()}')
raw.describe()

In [ ]:
# vehicle_health_score distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(raw['vehicle_health_score'], bins=40, color='steelblue', edgecolor='white')
plt.axvline(60, color='red', linestyle='--', label='Risk threshold (60)')
plt.title('Vehicle Health Score Distribution')
plt.xlabel('Health Score')
plt.legend()

plt.subplot(1, 2, 2)
at_risk = (raw['vehicle_health_score'] < 60).sum()
healthy = len(raw) - at_risk
plt.pie([healthy, at_risk], labels=['Healthy', 'At Risk'], colors=['green', 'red'],
        autopct='%1.1f%%', startangle=90)
plt.title('Class Distribution (threshold: 60)')
plt.tight_layout()
plt.show()

print(f'At-risk vehicles (score < 60): {at_risk} ({at_risk/len(raw)*100:.1f}%)')

## 2. Feature Engineering

The enriched CSV now includes real sensor readings (`engine_temp_c`, `tyre_pressure_kpa`, `vibration_g`, `km_since_service`) correlated with `vehicle_health_score`. We add a `health_band` categorical feature and a binary `at_risk` label.

In [ ]:
df = raw.copy()

# Binary label: health < 60 → at risk (already present as at_risk in CSV)
if "at_risk" not in df.columns:
    df["at_risk"] = (df["vehicle_health_score"] < 60).astype(int)

# Sensor columns are now real (from CSV) — no simulation needed
# Clip to valid physical ranges just in case
df["engine_temp_c"]     = df["engine_temp_c"].clip(80, 210)
df["tyre_pressure_kpa"] = df["tyre_pressure_kpa"].clip(145, 265)
df["vibration_g"]       = df["vibration_g"].clip(0, 2.5)
df["km_since_service"]  = df["km_since_service"].clip(0, 15000)

# Derived: health score band
df["health_band"] = pd.cut(
    df["vehicle_health_score"],
    bins=[0, 40, 60, 80, 100],
    labels=[3, 2, 1, 0]  # 3 = critical, 0 = healthy
).astype(int)

FEATURES = [
    "vehicle_health_score",
    "engine_temp_c",
    "tyre_pressure_kpa",
    "vibration_g",
    "km_since_service",
    "health_band",
]

print("Feature sample:")
df[FEATURES + ["at_risk"]].head()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df[FEATURES + ['at_risk']].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Train / Test Split + SMOTE

Class imbalance is common in maintenance datasets (failures are rare). SMOTE (Synthetic Minority Oversampling Technique) creates synthetic at-risk samples so the model doesn't just predict "healthy" for everything.

In [ ]:
X = df[FEATURES]
y = df['at_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train class balance: {y_train.value_counts().to_dict()}')

# Apply SMOTE to training set only
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'After SMOTE: {X_train_res.shape} | class balance: {pd.Series(y_train_res).value_counts().to_dict()}')

## 4. XGBoost Classifier

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
)

xgb.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test, y_test)],
    verbose=50,
)
print('Training complete.')

In [ ]:
y_pred      = xgb.predict(X_test)
y_prob      = xgb.predict_proba(X_test)[:, 1]
roc_auc     = roc_auc_score(y_test, y_prob)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Healthy', 'At Risk']))
print(f'ROC-AUC: {roc_auc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Healthy', 'At Risk'],
    ax=axes[0], colorbar=False
)
axes[0].set_title('Confusion Matrix')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, label=f'XGBoost (AUC={roc_auc:.3f})', color='steelblue')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importances = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color='steelblue')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 5. Cross-Validation

In [ ]:
cv_scores = cross_val_score(xgb, X, y, cv=5, scoring='recall')
print(f'5-fold CV Recall: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'Scores: {[round(s, 3) for s in cv_scores]}')
print()
print('Note: Recall is the priority metric — missing a real failure (false negative)')
print('costs 3× more than a false alarm (unnecessary service check).')

## 6. Save Model

Uses the `MaintenanceRiskModel` wrapper from `src/models/wrappers.py`:
```python
risk_scores = model.predict_proba(df)[:, 1]  # probability of failure
```
Accepts sensor columns from the CSV; imputes any missing ones from `vehicle_health_score`.

In [ ]:
import sys
sys.path.insert(0, ".")
from src.models.wrappers import MaintenanceRiskModel

maintenance_model = MaintenanceRiskModel(xgb_model=xgb, feature_cols=FEATURES)

test_input = pd.DataFrame([{"vehicle_health_score": 45.0}, {"vehicle_health_score": 85.0}])
probs = maintenance_model.predict_proba(test_input)[:, 1]
print("Sanity check:")
print(f"  Score 45 (at-risk)  → failure prob: {probs[0]:.3f}")
print(f"  Score 85 (healthy)  → failure prob: {probs[1]:.3f}")


In [ ]:
os.makedirs('src/models', exist_ok=True)
joblib.dump(maintenance_model, 'src/models/maintenance_model.pkl')
print('Saved → src/models/maintenance_model.pkl')

# Verify
loaded = joblib.load('src/models/maintenance_model.pkl')
v = loaded.predict_proba(test_input)[:, 1]
print(f'Load OK — probs: {v.round(3).tolist()}')

In [ ]:
# Model saved — commit to GitHub so HF Spaces has it:
#   git add src/models/
#   git commit -m 'Add trained model'
#   git push
print("Done. Commit and push src/models/ to GitHub.")

## Summary

| Metric | Score |
|--------|-------|
| ROC-AUC | see above |
| Recall (at-risk) | see above |
| 5-fold CV Recall | see above |

**Key finding:** `vehicle_health_score` + `engine_temp_c` are the top predictors of vehicle failure.  
**Next:** Run `03_fraud_detection.ipynb`.